# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [4]:
import os, glob
print("cwd:", os.getcwd())
print("local xgboost candidates:", glob.glob("xgboost*"))


cwd: C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project
local xgboost candidates: []


In [2]:
import sys
!{sys.executable} -m pip -V
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --no-cache-dir -U xgboost


C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Scripts\python.exe: No module named pip
C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Scripts\python.exe: No module named pip
C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Scripts\python.exe: No module named pip


In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [6]:

df = pd.read_csv("One-Hot-Encoded.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

MaxAge                                                                                       float64
AgeNum                                                                                       float64
WorkExp                                                                                      float64
YearsCode                                                                                    float64
RemoteCategoryNum                                                                            float64
                                                                                              ...   
AIModelsHaveWorkedWith__['openai gpt (chatbot models)', 'openai image generating models']      int64
AIModelsHaveWorkedWith__['openai gpt (chatbot models)', 'openai reasoning models']             int64
AIModelsHaveWorkedWith__['openai gpt (chatbot models)']                                        int64
AIModelsHaveWorkedWith__[]                                                                 

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [7]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0
    elif x <= 6:
        return 1
    else:
        return 2


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [8]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

Textspalten: ['LanguageWantToWorkWith', 'DatabaseWantToWorkWith', 'PlatformWantToWorkWith', 'WebframeWantToWorkWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsWantToWorkWith', 'AIAgent_Uses']
Numerische Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'CompTotal', 'ConvertedCompYearly', 'MainBranch_i am a developer by profession', 'MainBranch_i am learning to code', 'MainBranch_i am not primarily a developer, but i write code sometimes as part of my work/studies', 'MainBranch_i code primarily as a hobby', 'MainBranch_i used to be a developer by profession, but no longer am', 'MainBranch_i work with developers or my work supports developers but am not a developer by profession', 'MainBranch_nan', 'Age_18-24 years old', 'Age_25-34 years old', 'Age_35-44 years old', 'Age_45-54 years old', 'Age_55-64 years old', 'Age_nan', 'EdLevel_associate degree', 'EdLevel_bachelor’s degree', 'EdLevel_master’s degree', 'EdLevel_other', 'EdLevel_primar

,__text__,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,CompTotal,ConvertedCompYearly,MainBranch_i am a developer by profession,MainBranch_i am learning to code,...,AIModelsHaveWorkedWith__['anthropic: claude sonnet'],"AIModelsHaveWorkedWith__['gemini (flash general purpose models)', 'gemini (pro reasoning models)', 'openai gpt (chatbot models)']","AIModelsHaveWorkedWith__['gemini (flash general purpose models)', 'openai gpt (chatbot models)']",AIModelsHaveWorkedWith__['gemini (flash general purpose models)'],"AIModelsHaveWorkedWith__['openai gpt (chatbot models)', 'openai image generating models', 'openai reasoning models']","AIModelsHaveWorkedWith__['openai gpt (chatbot models)', 'openai image generating models']","AIModelsHaveWorkedWith__['openai gpt (chatbot models)', 'openai reasoning models']",AIModelsHaveWorkedWith__['openai gpt (chatbot models)'],AIModelsHaveWorkedWith__[],AIModelsHaveWorkedWith__other
0,['dart'] [] [] [] [] ['markdown file'] [] ['so...,34.0,29.0,8.0,14.0,0.00,52800.0,61256.0,1,0,...,0,0,0,0,1,0,0,0,0,0
1,"['java', 'python', 'swift'] ['dynamodb', 'mong...",34.0,29.0,2.0,10.0,0.25,90000.0,104413.0,1,0,...,0,0,0,0,0,0,0,1,0,0
3,"['java', 'kotlin'] [] ['amazon web services (a...",44.0,39.0,4.0,5.0,0.00,31200.0,36197.0,1,0,...,0,0,0,0,0,0,0,0,1,0
7,"['assembly', 'bash/shell (all shells)', 'html/...",44.0,39.0,22.0,30.0,0.00,72000.0,72000.0,1,0,...,0,0,0,0,0,0,0,1,0,0
8,"['scala'] ['dynamodb', 'mysql', 'postgresql'] ...",34.0,29.0,9.0,15.0,0.00,70000.0,70000.0,1,0,...,0,0,0,0,0,0,0,0,1,0


## Train/Test Split



In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 12520
Test size: 3131
Train class distribution:
 JobSat
2    0.719249
1    0.218930
0    0.061821
Name: proportion, dtype: float64
Test class distribution:
 JobSat
2    0.719259
1    0.219099
0    0.061642
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

preprocessor

,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit XGBoost-Feature-Importances)
- Klassifikator (XGBClassifier)


In [19]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))),
    ("classifier", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

pipeline


,steps,"[('preprocessing', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## GridSearchCV



In [20]:
parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # XGBoost: kompakter, sinnvoller Suchraum
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8],
    "classifier__reg_lambda": [1.0, 2.0],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

grid


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__colsample_bytree': [0.8], 'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [4, 6], 'classifier__n_estimators': [300, 500], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text', ...), ('num', ...)]"


## Grid Search + Beste Parameter


In [21]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=2.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=  19.2s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=1.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=  19.7s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=2.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=

## Evaluation auf Testdaten


In [23]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.17      0.01      0.02       193
           1       0.43      0.12      0.19       686
           2       0.75      0.97      0.84      2252

    accuracy                           0.72      3131
   macro avg       0.45      0.37      0.35      3131
weighted avg       0.64      0.72      0.65      3131

Confusion Matrix (rows=true, cols=pred):
[[   2   42  149]
 [   5   85  596]
 [   5   70 2177]]


In [24]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

           0       0.98      0.26      0.41       774
           1       0.81      0.23      0.35      2741
           2       0.77      0.99      0.87      9005

    accuracy                           0.78     12520
   macro avg       0.85      0.49      0.54     12520
weighted avg       0.79      0.78      0.73     12520



In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier

# -------------------------
# 0) Laden
# -------------------------
df = pd.read_csv("Abgaben/survey_results_after_clustering.csv")
print("Loaded:", df.shape)

target_col = "RemoteCategoryNum"

# -------------------------
# 1) Target vorbereiten
# -------------------------
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(float).round(2)

# Mapping 0..1 -> 0..4
y = (df[target_col] * 4).round().astype(int)  # 0..4
print("Target distribution:\n", y.value_counts().sort_index())

# ✅ Target aus Features entfernen
df = df.drop(columns=[target_col])

# -------------------------
# 2) Leakage-Spalten entfernen
# -------------------------
leak_cols = set()

# falls RemoteWork existiert (String-Spalte)
if "RemoteWork" in df.columns:
    leak_cols.add("RemoteWork")

# alles was "Remote" im Namen hat (außer Target, ist ja schon raus)
for c in df.columns:
    if "remote" in c.lower():
        leak_cols.add(c)

# optional: cluster rausnehmen (weil "nicht vom clustering ausgehen")
if "cluster" in df.columns:
    leak_cols.add("cluster")

# IDs raus
for maybe_id in ["ResponseId"]:
    if maybe_id in df.columns:
        leak_cols.add(maybe_id)

df = df.drop(columns=sorted(leak_cols), errors="ignore")
print("Dropped columns (leak/ids/etc):", sorted(leak_cols))
print("After drop:", df.shape)

# -------------------------
# 3) Features bestimmen
# -------------------------
X = df.copy()

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("Categorical cols:", len(cat_cols))
print("Numeric cols:", len(num_cols))
print("X shape:", X.shape, "y shape:", y.shape)

assert len(X) == len(y), f"Mismatch: X={len(X)} vs y={len(y)}"

# -------------------------
# 4) Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train), "Test size:", len(X_test))

# -------------------------
# 5) Preprocessing
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="drop"
)

# -------------------------
# 6) Modell + Pipeline
# -------------------------
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", xgb)
])

# -------------------------
# 7) GridSearch
# -------------------------
params = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [4, 6],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
    "clf__reg_lambda": [1.0, 2.0],
}

grid = GridSearchCV(
    pipeline,
    param_grid=params,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBeste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

# -------------------------
# 8) Evaluation
# -------------------------
best = grid.best_estimator_
y_pred = best.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2,3,4]))


Loaded: (14687, 40)
Target distribution:
 RemoteCategoryNum
0    4296
1    2620
2    3197
3    2738
4    1836
Name: count, dtype: int64
Dropped columns (leak/ids/etc): ['RemoteMissing', 'RemoteWork', 'ResponseId']
After drop: (14687, 36)
Categorical cols: 30
Numeric cols: 6
X shape: (14687, 36) y shape: (14687,)
Train size: 11749 Test size: 2938
Fitting 3 folds for each of 64 candidates, totalling 192 fits


KeyboardInterrupt: 